In [ ]:
!pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 18.9 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [ ]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json


FASHION_ITEM_TAGGING_PROMPT = """
You are a fashion item tagging assistant for e-commerce product images. Analyze the image and identify the visible clothing item, colors, patterns, materials, style attributes, and useful product tags.

Return only valid JSON. Do not include markdown, explanations, or extra commentary.

Use this exact JSON structure:

{
  "primary_item": null,
  "item_category": null,
  "colors": [],
  "pattern": null,
  "material_or_texture": null,
  "sleeve_length": null,
  "neckline": null,
  "fit": null,
  "style_attributes": [],
  "occasion": [],
  "season": [],
  "visible_details": [],
  "ecommerce_tags": []
}

Instructions:
- "primary_item" should name the main clothing item visible in the image, such as t-shirt, blouse, hoodie, jacket, dress, skirt, jeans, trousers, shorts, sweater, coat, or shoes.
- "item_category" should be a broad category such as top, bottom, outerwear, dress, footwear, accessory, or unknown.
- "colors" should list the main visible colors of the clothing item. Use simple color names such as black, white, blue, red, beige, gray, green, pink, brown, yellow, purple, or multicolor.
- "pattern" should describe the visible pattern, such as solid, striped, plaid, floral, graphic, polka dot, animal print, colorblock, textured, or unknown.
- "material_or_texture" should describe visible fabric or texture only when clear, such as denim, leather-like, knit, cotton-like, satin-like, fleece, ribbed, sheer, wool-like, linen-like, or unknown.
- "sleeve_length" should be one of: sleeveless, short sleeve, long sleeve, three-quarter sleeve, strapless, not_applicable, or unknown.
- "neckline" should be one of: crew neck, v-neck, scoop neck, collared, turtleneck, square neck, halter, strapless, not_applicable, or unknown.
- "fit" should describe the visible fit, such as slim, regular, relaxed, oversized, cropped, high-waisted, wide-leg, fitted, loose, or unknown.
- "style_attributes" should include visual style tags such as casual, formal, sporty, streetwear, minimalist, vintage, bohemian, elegant, business, athleisure, preppy, edgy, or classic.
- "occasion" should include likely e-commerce use cases such as everyday, work, party, athletic, formal, lounge, outdoor, beach, or unknown.
- "season" should include likely seasonal tags such as spring, summer, fall, winter, all-season, or unknown.
- "visible_details" should list notable visible details such as buttons, zipper, hood, pockets, drawstring, ruffles, pleats, embroidery, logo, graphic print, belt loops, cuffs, collar, or none.
- "ecommerce_tags" should provide concise searchable tags useful for online product filtering.

Only tag what is visually observable. Do not guess brand, price, gender identity, size, or personal attributes of the model. If an attribute is not visible or cannot be determined, use null, "unknown", or an empty list as appropriate.
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.fashion-item-tagging",
        description="Identify clothing items, colors, and style attributes for e-commerce auto-tagging",
        worker_release="qwen3-instruct",
        text_prompt=FASHION_ITEM_TAGGING_PROMPT,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=300,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a42c0e5b1f76ff8000bb7fbe390a85 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.describe.fashion-item-tagging', tag='1.0.1'), AbilityAliasEntry(alias='datasciencealliance-org.describe.fashion-item-tagging', tag='latest')]


### Evalulate on a Single Image

In [ ]:
from pathlib import Path
import json

pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.describe.fashion-item-tagging:latest"
    )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    sample_img_path = Path("/content/sample_fashion_image.jpg")

    job = endpoint.upload(sample_img_path)
    result = job.predict()

print("=== RAW EYEPOP RESULT ===")
print(json.dumps(result, indent=2))

print("\n=== FASHION ITEM TAGGING OUTPUT ===")
texts = result.get("texts", [])

if not texts:
    print("No text output found. Check that the image path is correct and the ability alias was published successfully.")
else:
    for text_item in texts:
        raw_text = text_item.get("text", "")

        try:
            parsed = json.loads(raw_text)
            print(json.dumps(parsed, indent=2))
        except json.JSONDecodeError:
            print(raw_text)

=== RAW EYEPOP RESULT ===
{
  "compute_units": 0.1,
  "seconds": 0,
  "source_height": 768,
  "source_id": "e6c6aa12-73ec-11f1-a2ff-fec23a935fa3",
  "source_width": 1408,
  "system_timestamp": 1782759688432882000,
  "texts": [
    {
      "id": 1,
      "text": "```json\n{\n  \"primary_item\": \"skirt\",\n  \"item_category\": \"bottom\",\n  \"colors\": [\n    \"yellow\"\n  ],\n  \"pattern\": \"solid\",\n  \"material_or_texture\": \"knit\",\n  \"sleeve_length\": \"not_applicable\",\n  \"neckline\": \"not_applicable\",\n  \"fit\": \"high-waisted\",\n  \"style_attributes\": [\n    \"elegant\",\n    \"classic\",\n    \"minimalist\"\n  ],\n  \"occasion\": [\n    \"everyday\",\n    \"work\",\n    \"formal\"\n  ],\n  \"season\": [\n    \"all-season\"\n  ],\n  \"visible_details\": [\n    \"pleats\",\n    \"waistband\"\n  ],\n  \"ecommerce_tags\": [\n    \"pleated skirt\",\n    \"midi skirt\",\n    \"high-waisted skirt\",\n    \"yellow skirt\",\n    \"solid skirt\",\n    \"work skirt\",\n    \"